<a href="https://colab.research.google.com/github/ahmadirhamm/toxic-comment-classification-youtube/blob/main/Project_UAS_NLP_kel_5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**PROJECT UJIAN AKHIR SEMESTER**

Mata Kuliah : NLP
=====================================
**KELOMPOK 5 (KLASIFIKASI KOMENTAR TOXIC)**

1. Ahmad Irham Dzulkifli - 2011102441211
2. Ramayasin Gymnastiar - 2011102441026
3. Tasya Putri Hardyani - 2011102441133

In [1]:
!pip install Sastrawi

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.7/209.7 kB 5.1 MB/s eta 0:00:00


In [3]:
import pandas as pd

# ===== 1. Membaca file dataset Excel =====
nama_file = "dataset_rakyat_ga_perlu_dolar.xlsx"
df = pd.read_excel(nama_file)

# ===== 2. Cek 5 data teratas =====
print(">>> 5 Data Teratas Dataset <<<")
df.head()

>>> 5 Data Teratas Dataset <<<


,Teks Komentar,Label,Keterangan
0,Menurutku ini bukan blunder tapi ketololan yan...,1.0,"Toxic (Mengandung kata ""ketololan"")"
1,Ngomong yang penting❌\nYang penting ngomong✅,0.0,Non-Toxic (Kritik netral/sarkasme halus)
2,"SETIAP PRABOWO PIDATO, KIAMAT MAJU SATU HARI.\...",0.0,Non-Toxic (Sarkasme/bercandaan netizen)
3,Seumur hidup saya tinggal di desa....!!\nDan s...,0.0,Non-Toxic (Curhatan/argumen warga desa)
4,Gila ini sekelas presiden loh😢 statement ngaco,1.0,"Toxic (Mengandung kata ""gila"" & ""ngaco"")"


**PREPROCESSING**

In [5]:
import re
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

# Menyiapkan stopword remover dari Sastrawi
factory = StopWordRemoverFactory()
stopword_remover = factory.create_stop_word_remover()

def bersihkan_teks(teks):
    # Pastikan data bertipe string (mencegah error jika ada baris kosong/nan)
    teks = str(teks)

    # Tahap 1: Case Folding (Mengubah ke huruf kecil semua)
    teks = teks.lower()

    # Tahap 2: Menghapus URL/Link internet
    teks = re.sub(r'https?://\S+|www\.\S+', '', teks)

    # Tahap 3: Menghapus Username YouTube (@nama_user) dan tanda hashtag (#)
    teks = re.sub(r'@\w+|#\w+', '', teks)

    # Tahap 4: Menghapus Angka dan Tanda Baca (Hanya menyisakan huruf alfabet dan spasi)
    teks = re.sub(r'[^a-zA-Z\s]', '', teks)

    # Tahap 5: Stopwords Removal (Menghapus kata tidak penting seperti: yang, di, ke, dari)
    teks = stopword_remover.remove(teks)

    # Tahap 6: Merapikan spasi yang ganda/berlebihan
    teks = teks.strip()
    teks = re.sub(r'\s+', ' ', teks)

    return teks

# disini nama kolom excel "Teks Komentar"
nama_kolom_teks = "Teks Komentar"

# Menerapkan fungsi pembersihan ke seluruh baris data secara instan
df["Komentar_Bersih"] = df[nama_kolom_teks].apply(bersihkan_teks)

print("--- Hasil Perbandingan Preprocessing ---")
# Menampilkan kolom mentah vs kolom bersih untuk pembuktian
df[[nama_kolom_teks, "Komentar_Bersih"]].head()

--- Hasil Perbandingan Preprocessing ---


,Teks Komentar,Komentar_Bersih
0,Menurutku ini bukan blunder tapi ketololan yan...,menurutku bukan blunder ketololan dipertontonk...
1,Ngomong yang penting❌\nYang penting ngomong✅,ngomong penting yang penting ngomong
2,"SETIAP PRABOWO PIDATO, KIAMAT MAJU SATU HARI.\...",prabowo pidato kiamat maju satu hari
3,Seumur hidup saya tinggal di desa....!!\nDan s...,seumur hidup tinggal desa dan rasa orang desa ...
4,Gila ini sekelas presiden loh😢 statement ngaco,gila sekelas presiden loh statement ngaco


**MEMBERSIHKAN DATA KOSONG (NaN)**

In [6]:
# --- MEMBERSIHKAN DATA KOSONG (NaN) ---

print("Jumlah data sebelum dibersihkan:", len(df))

# 1. Menghapus baris jika kolom 'Komentar Bersih' atau 'Label' ada yang kosong
df = df.dropna(subset=["Komentar_Bersih", "Label"])

# 2. Memastikan kolom Label bertipe integer (0 atau 1) tanpa desimal
df["Label"] = df["Label"].astype(int)

print("Jumlah data setelah dibersihkan:", len(df))

Jumlah data sebelum dibersihkan: 1091
Jumlah data setelah dibersihkan: 1069


In [7]:
print(df['Label'].value_counts())

Label
0    831
1    238
Name: count, dtype: int64


**Feature Extraction (TF-IDF) & Split Data**

In [8]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

# 1. Menentukan kolom Fitur (X) dan kolom Target/Label (y)
# disini nama kolom excel "Komentar_Bersih"
nama_kolom_bersih = "Komentar_Bersih"

X = df[nama_kolom_bersih]
y = df["Label"] # Kolom "Label" yang berisi angka 0 dan 1 dari Excel

# 2. Split Dataset: 80% untuk Latih Model, 20% untuk Uji Akurasi
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. Ekstraksi Fitur menggunakan TF-IDF Vectorizer
# Fungsi ini akan otomatis menghitung bobot nilai dari setiap kata unik
tfidf = TfidfVectorizer()

# Latih TF-IDF menggunakan data teks latih, lalu transformasikan menjadi matriks angka
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print("--- Proses Ekstraksi Fitur Selesai ---")
print(f"Jumlah Data Latih (X_train): {X_train_tfidf.shape[0]} baris")
print(f"Jumlah Data Uji (X_test)  : {X_test_tfidf.shape[0]} baris")
print(f"Total Kosakata Unik       : {X_train_tfidf.shape[1]} kata")

--- Proses Ekstraksi Fitur Selesai ---
Jumlah Data Latih (X_train): 855 baris
Jumlah Data Uji (X_test)  : 214 baris
Total Kosakata Unik       : 3057 kata


**TRAINING & EVALUASI MODEL**

In [ ]:
from sklearn.metrics import classification_report, accuracy_score
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import SVC

# ==========================================
# 1. PELATIHAN MODEL ALGORITMA 1: NAIVE BAYES
# ==========================================
model_nb = MultinomialNB()
model_nb.fit(X_train_tfidf, y_train)

# Prediksi menggunakan data uji
y_pred_nb = model_nb.predict(X_test_tfidf)


# ==========================================
# 2. PELATIHAN MODEL ALGORITMA 2: SUPPORT VECTOR MACHINE (SVM)
# ==========================================
model_svm = SVC(kernel='linear') # Menggunakan kernel linear untuk teks klasifikasi
model_svm.fit(X_train_tfidf, y_train)

# Prediksi menggunakan data uji
y_pred_svm = model_svm.predict(X_test_tfidf)


# ==========================================
# 3. OUTPUT EVALUASI PERBANDINGAN TUGAS KULIAH
# ==========================================
print("=======================================================")
print("     HASIL EVALUASI MODEL KLASIFIKASI KOMENTAR TOXIC   ")
print("=======================================================\n")

print(f"-> Akurasi Model Naive Bayes : {accuracy_score(y_test, y_pred_nb) * 100:.2f}%")
print(f"-> Akurasi Model SVM         : {accuracy_score(y_test, y_pred_svm) * 100:.2f}%\n")

print("--- DETAIL PERFORMA MODEL SVM (TERBAIK) ---")
# Classification report memberikan info Precision, Recall, dan F1-Score untuk bab 4 laporan Anda
print(classification_report(y_test, y_pred_svm, target_names=['Non-Toxic (0)', 'Toxic (1)']))

     HASIL EVALUASI MODEL KLASIFIKASI KOMENTAR TOXIC   

-> Akurasi Model Naive Bayes : 78.97%
-> Akurasi Model SVM         : 83.64%

--- DETAIL PERFORMA MODEL SVM (TERBAIK) ---
               precision    recall  f1-score   support

Non-Toxic (0)       0.84      0.97      0.90       167
    Toxic (1)       0.77      0.36      0.49        47

     accuracy                           0.84       214
    macro avg       0.81      0.67      0.70       214
 weighted avg       0.83      0.84      0.81       214



**Analisis Hasil**

Perbandingan Model:

Support Vector Machine (SVM) dengan kernel linear terbukti lebih tangguh ($83.64\%$) dalam memetakan karakteristik teks komentar toxic netizen YouTube dibanding Naive Bayes ($78.97\%$).

---

Nilai Recall untuk Kelas Toxic (1):

Untuk kelas Toxic di tabel hanya 0.36 (36%).Artinya, model masih sering kecolongan mendeteksi komentar toxic (banyak komentar toxic yang dikira non-toxic oleh mesin).Kenapa bisa begitu?
ini terjadi karena data Imbalance (jumlah sampel Non-Toxic jauh lebih banyak, yaitu 167 data uji, dibandingkan data Toxic yang hanya 47 data uji). Selain itu, netizen sering menggunakan kata sindiran (sarkasme) atau kata makian yang disamarkan/salah ketik, sehingga TF-IDF belum mengenali polanya secara maksimal.

**FUNCTION DEMO: UJI COBA PREDIKSI KOMENTAR BARU**

In [ ]:
# =====================================================================
# FUNCTION DEMO: UJI COBA PREDIKSI KOMENTAR BARU
# =====================================================================

def prediksi_komentar_baru(teks_input):
    # 1. Bersihkan teks input menggunakan fungsi preprocessing yang sudah dibuat sebelumnya
    teks_bersih = bersihkan_teks(teks_input)

    # 2. Ubah teks menjadi angka menggunakan vectorizer TF-IDF yang sudah terlatih
    teks_tfidf = tfidf.transform([teks_bersih])

    # 3. Prediksi menggunakan model SVM (Terbaik)
    prediksi = model_svm.predict(teks_tfidf)[0]

    # 4. Tampilkan Hasil
    status = "⚠️ TOXIC" if prediksi == 1 else "✅ NON-TOXIC"
    print(f"Komentar : '{teks_input}'")
    print(f"Hasil    : {status}\n")

# --- Silakan Tes Ketik Komentar Bebas di Sini Untuk Pembuktian ---
prediksi_komentar_baru("Keren banget kontennya bang, sangat mengedukasi!")
prediksi_komentar_baru("Halah bacot amat lu, konten sampah begini mending di-report aja!")

Komentar : 'Keren banget kontennya bang, sangat mengedukasi!'
Hasil    : ✅ NON-TOXIC

Komentar : 'Halah bacot amat lu, konten sampah begini mending di-report aja!'
Hasil    : ✅ NON-TOXIC



**CATATAN HASIL FUNCION DEMO KOMENTAR BARU :**
===
Secara logika manusia, kata "bacot" dan "sampah" jelas kasar dan masuk kategori toxic. Namun, kenapa model SVM justru melabelinya sebagai Non-Toxic?

Ada 3 alasan ilmiah dan teknis yang menyebabkan model SVM salah memprediksi komentar baru:

1. Dampak dari Data Imbalance (Data Tidak Seimbang)
Jika dilihat kembali data evaluasi:

- Sampel data Non-Toxic (0) jauh lebih mendominasi, yaitu ada 167 data.

- Sampel data Toxic (1) sangat sedikit, hanya 47 data.

Karena data Non-Toxic jauh lebih banyak, model SVM menjadi "malas" dan memiliki kecenderungan kuat untuk menebak segala sesuatu sebagai Non-Toxic agar cari aman (nilai Recall kelas toxic rendah, hanya 0.36). Modelnya kurang belajar mengenali pola kata toxic karena contohnya di dataset sedikit.

2. Efek Tahap Stopwords Removal
Di dalam fungsi bersihkan_teks, kita menggunakan StopWordRemoverFactory bawaan Sastrawi. Mari kita bedah apa yang terjadi pada komentar kedua setelah dibersihkan:

- Teks Mentah: "Halah bacot amat lu, konten sampah begini mending di-report aja!"

- Setelah Preprocessing: "halah bacot amat lu konten sampah mending report aja" (atau kata seperti di, begini hilang).

Meskipun kata "bacot" tetap ada, kata tersebut dikelilingi oleh kata-kata umum seperti "konten", "mending", "report", "aja". Jika di dalam 872 data latih, kata-kata umum ini lebih sering muncul di komentar berlabel 0 (Non-Toxic), maka bobot TF-IDF akan menyeret kalimat tersebut ke arah Non-Toxic.

3. Kata "Bacot" Mungkin Belum Cukup Kuat di Data Latih (TF-IDF Weight)
Sistem TF-IDF bekerja berdasarkan frekuensi. Kemungkinan besar:

- Kata "bacot" jarang muncul atau bahkan tidak ada di dalam 872 baris data yang dipakai untuk melatih mesin.

- Jika sebuah kata asing atau jarang ditemukan saat training, model tidak akan tahu bahwa kata itu bermakna kasar, sehingga model SVM hanya menebak berdasarkan kata-kata lain di sekitarnya yang dianggapnya "aman".

**Catatan Tambahan**

Ada 2 cara untuk memperbaiki ini:
- Gunakan Kamus Slang Mandiri (Normalisasi): Netizen sering menulis bacot, bct, bacott. Bisa membuat dictionary untuk mengubah kata slang kasar menjadi kata baku toxic sebelum masuk ke TF-IDF.

- Menambah Data Toxic (Oversampling): Mencari tambahan khusus komentar yang toxic, agar jumlah data 0 dan 1 menjadi seimbang ($50:50$).